# Movie Graph – Advanced Data Models


## Anleitung:
1. Diese Notebook-Kopie öffnen
2. In der Zelle unten URI, Username, Passwort und DatenbanDatenbank eintragen
3. Zellen von oben nach unten ausführen
4. Aufgaben im Übungsblock lösen

## Installation der benötigten Paket

In [ ]:
!pip -q install neo4j pandas

## Zugangsdaten konfigurieren und testen

In [ ]:
URI = "URI"
USERNAME = "USERNAME"
PASSWORD = "PASSWORD"
DATABASE = "DATABASE"

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()

print("Verbindung erfolgreich.")

Verbindung erfolgreich.


### Hilfsfunktion für Cypher definieren

In [ ]:
def run_query(query, parameters=None, database=DATABASE):
    with driver.session(database=database) as session:
        result = session.run(query, parameters or {})
        return pd.DataFrame([record.data() for record in result])

In [ ]:
run_query("""
MATCH (m:Movie)
RETURN count(m) AS movies
""")

,movies
0,38


Erwartetes Ergebnis:

|index|movies|
|---|---|
|0|38|

In [ ]:
run_query("""
MATCH (p:Person)
RETURN count(p) AS persons
""")

,persons
0,133


Erwartetes Ergebnis:

|index|persons|
|---|---|
|0|133|

In [ ]:
run_query("""
MATCH ()-[r]->()
RETURN count(r) AS relationships
""")

,relationships
0,253


Erwartetes Ergebnis:

|index|relationships|
|---|---|
|0|253|

## Cypher

### Keywords

**`MATCH`**  
Mit `MATCH` beschreibst du das Muster, das im Graphen gesucht werden soll.  
Beispiel: `MATCH (p:Person)-[:ACTED_IN]->(m:Movie)` sucht Personen, die in einem Film mitgespielt haben.

**`WHERE`**  
`WHERE` schränkt Treffer ein.  
Damit kannst du Eigenschaften filtern oder Bedingungen formulieren.  
Beispiel: `WHERE m.title = 'The Matrix'`

**`RETURN`**  
`RETURN` legt fest, was am Ende ausgegeben wird.  
Du kannst Variablen, einzelne Properties oder berechnete Werte zurückgeben.

**`ORDER BY`**  
Mit `ORDER BY` sortierst du das Ergebnis.  
Mit `DESC` sortierst du absteigend.

**`LIMIT`**  
`LIMIT` begrenzt die Anzahl der zurückgegebenen Zeilen.

**`OPTIONAL MATCH`**  
`OPTIONAL MATCH` funktioniert ähnlich wie `MATCH`, liefert aber auch dann eine Zeile, wenn ein Teil des Musters fehlt.  
Fehlende Werte werden dann als `null` zurückgegeben.

**`WITH`**  
`WITH` verbindet mehrere Query-Teile miteinander.  
Damit kannst du Zwischenergebnisse weiterreichen, umbenennen oder filtern.

**`UNWIND`**  
`UNWIND` macht aus einer Liste mehrere Zeilen.  
Das ist nützlich, wenn du mit Listen von Werten arbeitest.

**`CREATE`**  
`CREATE` legt neue Knoten und Beziehungen an.

**`MERGE`**  
`MERGE` stellt sicher, dass ein Muster existiert.  
Wenn es schon vorhanden ist, wird es gefunden.  
Wenn nicht, wird es angelegt.

**`SET`**  
`SET` setzt oder verändert Properties oder Labels.

**`REMOVE`**  
`REMOVE` entfernt Properties oder Labels.

**`DELETE`**  
`DELETE` löscht Knoten oder Beziehungen.  
Knoten mit bestehenden Beziehungen lassen sich nicht direkt löschen.

**`DETACH DELETE`**  
`DETACH DELETE` löscht einen Knoten zusammen mit allen dazugehörigen Beziehungen.

### Aufbau einer Query
Eine typische Cypher-Query liest sich oft so:

- `MATCH` das Muster
- `WHERE` die Bedingung
- `RETURN` das Ergebnis

Bei Schreibabfragen kommen oft noch `CREATE`, `MERGE` und `SET` dazu.


### Mini-Beispiel

```cypher
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
WHERE m.title = 'The Matrix'
RETURN p.name AS actor
ORDER BY actor
LIMIT 10

## Übungsblock

### Aufgabe 1 - Wer hat einen bestimmten Film inszeniert?

Schreibe eine Query, die die Regisseurinnen oder Regisseure von "Cloud Atlas" ausgibt.


Tipp:

Gesucht sind Personen mit einer DIRECTED-Beziehung auf den Film Cloud Atlas.


In [ ]:
run_query("""
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title:'Cloud Atlas'})
RETURN p.name AS director
ORDER BY director
""")

,director
0,Lana Wachowski
1,Lilly Wachowski
2,Tom Tykwer


Erwartetes Ergebnis:

|index|director|
|---|---|
|0|Lana Wachowski|
|1|Lilly Wachowski|
|2|Tom Tykwer|

---

### Aufgabe 2 – Welche Filme sind über gemeinsame Schauspieler mit „Top Gun“ verbunden?

Schreibe eine Query, die Filme findet, die über gemeinsame Schauspieler mit "Top Gun" verbunden sind.
Gib zusätzlich aus, wie viele gemeinsame Schauspieler gefunden wurden.

Tipp:

Starte bei Top Gun, gehe zu den Schauspielern und von dort weiter zu anderen Filmen.

Wichtig: other <> m, damit Top Gun nicht selbst zurückkommt.

In [ ]:
run_query("""
MATCH (m:Movie {title:'Top Gun'})<-[:ACTED_IN]-(p:Person)-[:ACTED_IN]->(other:Movie)
WHERE other <> m
RETURN other.title AS movie,
       count(DISTINCT p) AS numSharedActors,
       collect(DISTINCT p.name) AS sharedActors
ORDER BY numSharedActors DESC, movie
""")

,movie,numSharedActors,sharedActors
0,A Few Good Men,1,[Tom Cruise]
1,Jerry Maguire,1,[Tom Cruise]
2,Joe Versus the Volcano,1,[Meg Ryan]
3,Sleepless in Seattle,1,[Meg Ryan]
4,When Harry Met Sally,1,[Meg Ryan]
5,You've Got Mail,1,[Meg Ryan]


Erwartetes Ergebnis:

|index|movie|numSharedActors|sharedActors|
|---|---|---|---|
|0|A Few Good Men|1|Tom Cruise|
|1|Jerry Maguire|1|Tom Cruise|
|2|Joe Versus the Volcano|1|Meg Ryan|
|3|Sleepless in Seattle|1|Meg Ryan|
|4|When Harry Met Sally|1|Meg Ryan|
|5|You've Got Mail|1|Meg Ryan|

---

### Aufgabe 3 – Welche Personen folgen Jessica Thompson?
Schreibe eine Query, die alle Personen ausgibt, die Jessica Thompson folgen.

Tipp:

Achte auf die Richtung von FOLLOWS.


In [ ]:
run_query("""
MATCH (p:Person)-[:FOLLOWS]->(:Person {name:'Jessica Thompson'})
RETURN p.name AS follower
ORDER BY follower
""")

,follower
0,Angela Scope
1,James Thompson


Erwartetes Ergebnis:

|index|follower|
|---|---|
|0|Angela Scope|
|1|James Thompson|

---

### Aufgabe 4 – Welche Filme hat Jessica Thompson reviewed?
Gib Filmtitel, Summary und Rating von Jessica Thompsons Reviews aus.

In [ ]:
run_query("""
MATCH (:Person {name:'Jessica Thompson'})-[r:REVIEWED]->(m:Movie)
RETURN m.title AS movie,
       r.summary AS summary,
       r.rating AS rating
ORDER BY rating DESC, movie
""")

,movie,summary,rating
0,Cloud Atlas,An amazing journey,95
1,Jerry Maguire,You had me at Jerry,92
2,Unforgiven,"Dark, but compelling",85
3,The Da Vinci Code,A solid romp,68
4,The Replacements,"Silly, but fun",65
5,The Birdcage,Slapstick redeemed only by the Robin Williams ...,45


Erwartetes Ergebnis:

|index|movie|summary|rating|
|---|---|---|---|
|0|Cloud Atlas|An amazing journey|95|
|1|Jerry Maguire|You had me at Jerry|92|
|2|Unforgiven|Dark, but compelling|85|
|3|The Da Vinci Code|A solid romp|68|
|4|The Replacements|Silly, but fun|65|
|5|The Birdcage|Slapstick redeemed only by the Robin Williams and Gene Hackman's stellar performances|45|

---

### Aufgabe 5 – Zusatz: einfache Empfehlung
Finde Filme, die James Thompson noch nicht reviewed hat, die aber von Personen reviewed wurden, die mindestens einen Film mit ihm gemeinsam reviewed haben.

Tipp:

Das Problem hat drei Schritte:
1. James reviewed einen Film
2. Andere Personen reviewed denselben Film
3. Diese anderen Personen reviewed noch weitere Filme

Diese weiteren Filme dürfen nicht schon von James reviewed worden sein

In [ ]:
run_query("""
MATCH (j:Person {name:'James Thompson'})-[:REVIEWED]->(common:Movie)<-[:REVIEWED]-(other:Person)
MATCH (other)-[r:REVIEWED]->(rec:Movie)
WHERE other <> j
  AND NOT EXISTS {
    MATCH (j)-[:REVIEWED]->(rec)
  }
RETURN rec.title AS recommendation,
       collect(DISTINCT other.name) AS recommendedBy,
       round(avg(r.rating),1) AS avgRating
ORDER BY avgRating DESC, recommendation
""")

,recommendation,recommendedBy,avgRating
0,Cloud Atlas,[Jessica Thompson],95.0
1,Jerry Maguire,[Jessica Thompson],92.0
2,Unforgiven,[Jessica Thompson],85.0
3,The Birdcage,[Jessica Thompson],45.0


Erwartetes Ergebnis:

|index|recommendation|recommendedBy|avgRating|
|---|---|---|---|
|0|Cloud Atlas|Jessica Thompson|95\.0|
|1|Jerry Maguire|Jessica Thompson|92\.0|
|2|Unforgiven|Jessica Thompson|85\.0|
|3|The Birdcage|Jessica Thompson|45\.0|

## Ressourcen freigeben

In [ ]:
driver.close()
print("Driver geschlossen.")

Driver geschlossen.
